# 08a — Export the trajectory Seurat object for RNA velocity

Exports the annotated `20250526p38-draw/sub/2stsub.rds` object to sparse matrices and metadata consumed by notebook 08b. Source audited against `20250526p38-draw/sub/velocity.ipynb` (cell 1).


In [ ]:
# Centralized paths and scheduler-aware thread limits.
repo_root <- if (file.exists("config/paths.R")) "." else if (file.exists("../config/paths.R")) ".." else stop("Run Jupyter from the repository root or notebooks/ directory.")
source(file.path(repo_root, "config", "paths.R"))


In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
})

input_rds <- p38_path("20250526p38-draw", "sub", "2stsub.rds")
export_dir <- p38_path("20250526p38-draw", "sub", "dynamo")
dir.create(export_dir, recursive = TRUE, showWarnings = FALSE)
stopifnot(file.exists(input_rds))

combined <- readRDS(input_rds)
required_metadata <- c("orig.ident", "celltype")
missing_metadata <- setdiff(required_metadata, colnames(combined@meta.data))
if (length(missing_metadata)) stop("Missing Seurat metadata: ", paste(missing_metadata, collapse = ", "))

combined$seurat_clusters <- combined$celltype
DefaultAssay(combined) <- "RNA"
if ("layers" %in% slotNames(combined@assays$RNA)) combined <- JoinLayers(combined)
rownames(combined) <- make.unique(rownames(combined))

sample_id <- as.character(combined$orig.ident)
if (any(!sample_id %in% c("WT", "KO"))) stop("Unexpected orig.ident: ", paste(setdiff(unique(sample_id), c("WT", "KO")), collapse = ", "))
core_barcode <- sub("^(WT_|KO_)", "", colnames(combined))
new_barcode <- paste0(sample_id, "_", core_barcode)
if (anyDuplicated(new_barcode)) stop("Sample-prefixed barcodes are not unique.")
combined <- RenameCells(combined, new.names = new_barcode)

rna_layers <- Layers(combined[["RNA"]])
counts_data <- if ("counts" %in% rna_layers) LayerData(combined, assay = "RNA", layer = "counts") else GetAssayData(combined, assay = "RNA", slot = "counts")
norm_data <- if ("data" %in% rna_layers) LayerData(combined, assay = "RNA", layer = "data") else GetAssayData(combined, assay = "RNA", slot = "data")
stopifnot(identical(dim(counts_data), dim(norm_data)), ncol(counts_data) == ncol(combined))

writeMM(counts_data, file.path(export_dir, "matrix_counts.mtx"))
writeMM(norm_data, file.path(export_dir, "matrix_normalized.mtx"))
write.csv(data.frame(gene_name = rownames(combined)), file.path(export_dir, "genes.csv"), row.names = FALSE, quote = FALSE)
write.csv(data.frame(barcode = colnames(combined)), file.path(export_dir, "barcodes.csv"), row.names = FALSE, quote = FALSE)
write.csv(combined@meta.data, file.path(export_dir, "metadata.csv"), quote = FALSE)

if ("pca" %in% names(combined@reductions)) {
  write.csv(Embeddings(combined, "pca"), file.path(export_dir, "pca.csv"), quote = FALSE)
  write.csv(data.frame(stdev = Stdev(combined, "pca")), file.path(export_dir, "pca_stdev.csv"), row.names = FALSE, quote = FALSE)
}
if ("umap" %in% names(combined@reductions)) {
  write.csv(Embeddings(combined, "umap"), file.path(export_dir, "umap.csv"), quote = FALSE)
}

message("Exported ", ncol(combined), " cells and ", nrow(combined), " genes to ", export_dir)
